In [55]:
import os
import glob
import time
import shutil
from bs4 import BeautifulSoup
from DrissionPage import ChromiumPage, ChromiumOptions


import re
import csv
import warnings


In [56]:
def find_linux_browser_path():
    # 1. Daftar nama executable browser berbasis Chromium yang mungkin Anda miliki
    chromium_browsers = [
        "brave-browser",          # Brave Browser
        "microsoft-edge-stable",  # MS Edge
        "microsoft-edge",         # MS Edge
        "google-chrome",          # Google Chrome
        "chromium-browser",       # Chromium
        "chromium"
    ]
    
    # Cek via sistem PATH bawaan
    for binary in chromium_browsers:
        path = shutil.which(binary)
        if path:
            return path

    # 2. Cek lokasi spesifik instalasi Brave / Edge di Ubuntu (termasuk Snap/Flatpak/Opt)
    possible_paths = [
        # Path Brave
        "/usr/bin/brave-browser",
        "/snap/bin/brave",
        # Path Microsoft Edge
        "/usr/bin/microsoft-edge",
        "/usr/bin/microsoft-edge-stable",
        "/snap/bin/microsoft-edge",
 
    ]
    
    for path in possible_paths:
        if os.path.exists(path):
            return path

    return None

In [57]:
def init_drission_page():
    browser_path = find_linux_browser_path()

    if not browser_path:
        raise FileNotFoundError(
            "Browser Chrome/Chromium tidak ditemukan.\n"
            "Silakan jalankan perintah ini di terminal Ubuntu Anda:\n"
            "sudo apt update && sudo apt install -y chromium-browser"
        )

    print(f"[+] Lokasi browser ditemukan: {browser_path}")

    co = ChromiumOptions()
    co.set_browser_path(browser_path)
    
    # Inisialisasi ChromiumPage
    page = ChromiumPage(co)
    return page

In [58]:
def extract_links_from_html(html_source):
    soup = BeautifulSoup(html_source, 'html.parser')
    found_links = set()
    target_prefix = "https://putusan3.mahkamahagung.go.id/direktori/putusan/"
    
    # 1. Fokus hanya pada area popular-post-list-sidebar
    main_container = soup.find('div', id='popular-post-list-sidebar')
    
    if not main_container:
        # Jika kontainer tidak ditemukan (misal halaman lambat dimuat), berikan info
        print("   [!] Peringatan: Kontainer daftar putusan tidak ditemukan di halaman ini.")
        return []
        
    # 2. Cari semua blok putusan menggunakan CSS Selector untuk class "spost" dan "clearfix"
    putusan_blocks = main_container.select('div.spost.clearfix')
    
    for block in putusan_blocks:
        # 3. Cek badge Unpublish (Cari class yang mengandung 'badge-danger')
        unpublish_badge = block.select_one('.badge-danger')
        if unpublish_badge and "unpublish" in unpublish_badge.text.lower():
            continue  # Abaikan blok ini jika ada badge unpublish
            
        # 4. Cari link spesifik di dalam tag <strong> -> <a>
        link_tag = block.select_one('strong a')
        
        if link_tag and link_tag.has_attr('href'):
            href = link_tag['href']
            # Pastikan format link benar
            if href.startswith(target_prefix):
                found_links.add(href)
                
    return list(found_links)

In [59]:
# Menyimpan Link ke File .txt

def append_links_to_file(links, filename):
    if not links:
        return
    with open(filename, 'a', encoding='utf-8') as f:
        for link in links:
            f.write(f"{link}\n")
    print(f"   [+] {len(links)} link berhasil disimpan ke '{filename}'.")

In [60]:
# Pengecekan Cloudflare

def handle_cloudflare(page):
    """Menunggu pengguna menyelesaikan verifikasi Cloudflare jika muncul."""
    if "just a moment" in page.title.lower() or "attention required" in page.title.lower():
        print("   [!] Verifikasi Cloudflare terdeteksi. Silakan centang 'Saya Bukan Robot' di browser...")
        while "just a moment" in page.title.lower() or "attention required" in page.title.lower():
            time.sleep(2)
        print("   [✓] Verifikasi Cloudflare berhasil dilewati!")

In [61]:
# Fungsi Utama Program

def main():
    start_page = 41
    end_page = 60
    output_filename = "listURLPNSurabayaPage41-60.txt"
    base_url = "https://putusan3.mahkamahagung.go.id/pengadilan/profil/pengadilan/pn-surabaya/page/"

    # Reset file output
    open(output_filename, 'w', encoding='utf-8').close()

    print("Membuka Browser via DrissionPage (Bypass Cloudflare)...")
    page = init_drission_page()
    total_link = 0

    try:
        for page_num in range(start_page, end_page + 1):
            url = f"{base_url}{page_num}.html"
            print(f"\n==========================================")
            print(f"[+] Membuka Halaman {page_num}: {url}")
            print(f"==========================================")
            
            page.get(url)
            
            # Cek Cloudflare
            handle_cloudflare(page)
            
            # Waktu jeda agar konten JavaScript termuat
            time.sleep(3)
            
            # Ambil dan simpan link
            links = extract_links_from_html(page.html)
            append_links_to_file(links, output_filename)
            total_link += len(links)

    except Exception as e:
        print(f"\n[-] Terjadi kesalahan: {e}")
        
    finally:
        print("\nMenutup browser...")
        page.quit()
        print(f"\n== SELESAI == Total {total_link} link tersimpan di '{output_filename}'.")

In [62]:
if __name__ == "__main__":
    main()

Membuka Browser via DrissionPage (Bypass Cloudflare)...
[+] Lokasi browser ditemukan: /usr/bin/brave-browser

[+] Membuka Halaman 41: https://putusan3.mahkamahagung.go.id/pengadilan/profil/pengadilan/pn-surabaya/page/41.html
   [+] 18 link berhasil disimpan ke 'listURLPNSurabayaPage41-60.txt'.

[+] Membuka Halaman 42: https://putusan3.mahkamahagung.go.id/pengadilan/profil/pengadilan/pn-surabaya/page/42.html
   [+] 20 link berhasil disimpan ke 'listURLPNSurabayaPage41-60.txt'.

[+] Membuka Halaman 43: https://putusan3.mahkamahagung.go.id/pengadilan/profil/pengadilan/pn-surabaya/page/43.html
   [+] 18 link berhasil disimpan ke 'listURLPNSurabayaPage41-60.txt'.

[+] Membuka Halaman 44: https://putusan3.mahkamahagung.go.id/pengadilan/profil/pengadilan/pn-surabaya/page/44.html
   [+] 18 link berhasil disimpan ke 'listURLPNSurabayaPage41-60.txt'.

[+] Membuka Halaman 45: https://putusan3.mahkamahagung.go.id/pengadilan/profil/pengadilan/pn-surabaya/page/45.html
   [+] 19 link berhasil disimpa

## getMETAINFO.ipynb

In [70]:
# Konfigurasi Browser (Sama dengan skrip sebelumnya)

def find_linux_browser_path():
    chromium_browsers = ["brave-browser", "microsoft-edge-stable", "microsoft-edge", "google-chrome", "chromium-browser", "chromium"]
    for binary in chromium_browsers:
        path = shutil.which(binary)
        if path: return path

    possible_paths = ["/usr/bin/brave-browser", "/snap/bin/brave", "/usr/bin/microsoft-edge", "/usr/bin/google-chrome", "/usr/bin/chromium-browser"]
    for path in possible_paths:
        if os.path.exists(path): return path
    return None

def init_drission_page():
    browser_path = find_linux_browser_path()
    if not browser_path: raise FileNotFoundError("Browser tidak ditemukan.")
    co = ChromiumOptions()
    co.set_browser_path(browser_path)
    return ChromiumPage(co)

def handle_cloudflare(page):
    if "just a moment" in page.title.lower() or "attention required" in page.title.lower():
        print("   [!] Verifikasi Cloudflare terdeteksi...")
        while "just a moment" in page.title.lower() or "attention required" in page.title.lower():
            time.sleep(2)

In [76]:
def generateFileCSV(listHasil, csvName):
    # Didefinisikan kembali header dengan tambahan link_putusan
    headers = ("terdakwa", "penuntut_umum", "nomor", "tingkat_proses", "klasifikasi", "kata_kunci", "tahun", "tanggal_register",
               "lembaga_peradilan", "jenis_lembaga_peradilan", "hakim_ketua", "hakim_anggota", "panitera", "amar",
               "amar_lainnya", "catatan_amar", "tanggal_musyawarah", "tanggal_dibacakan", "kaidah", "abstrak", "url_pdf", "link_putusan")
               
    if os.path.exists(csvName):
        f = open(csvName, 'a', newline='\n', encoding='utf-8')
        w = csv.writer(f)
    else:
        f = open(csvName, 'w', newline='\n', encoding='utf-8')
        w = csv.writer(f)
        w.writerow(headers)

    for s in listHasil:
        w.writerow(s)

    f.close()
    print("\n[+] Create Csv file Berhasil")

In [75]:
def generateMeta(html_source, current_url=""):
    soup = BeautifulSoup(html_source, 'html.parser')
    
    # 1. Siapkan struktur data dengan kolom tambahan link_putusan
    fields = ["terdakwa", "penuntut_umum", "nomor", "tingkat_proses", "klasifikasi", "kata_kunci", "tahun", "tanggal_register",
              "lembaga_peradilan", "jenis_lembaga_peradilan", "hakim_ketua", "hakim_anggota", "panitera", "amar",
              "amar_lainnya", "catatan_amar", "tanggal_musyawarah", "tanggal_dibacakan", "kaidah", "abstrak", "url_pdf", "link_putusan"]
              
    data = {f: "" for f in fields}
    
    # Masukkan URL yang sedang diproses ke dalam kolom link_putusan
    data["link_putusan"] = current_url
    
    # 2. Cari tabel utama yang menyimpan metainfo
    table = soup.find('table', class_='table')
    
    if table:
        trs = table.find_all('tr')
        for tr in trs:
            tds = tr.find_all('td')
            
            # --- BAGIAN A: Menangani Nama Pihak (Terdakwa / Pemohon / Penggugat dll) ---
            if len(tds) == 1 and tds[0].has_attr('colspan'):
                title_pihak = tds[0].find('span', id='title_pihak')
                if title_pihak:
                    pihak_text = title_pihak.get_text(separator='|')
                    parts = [p.strip() for p in pihak_text.split('|') if p.strip()]
                    
                    current_role = None
                    penuntut_list = []
                    terdakwa_list = []
                    
                    for part in parts:
                        lower_part = part.lower()
                        if any(x in lower_part for x in ['penuntut umum', 'penggugat', 'pemohon']):
                            current_role = 'penuntut'
                        elif any(x in lower_part for x in ['terdakwa', 'tergugat', 'termohon']):
                            current_role = 'terdakwa'
                        else:
                            if current_role == 'penuntut':
                                penuntut_list.append(part)
                            elif current_role == 'terdakwa':
                                terdakwa_list.append(part)
                                
                    data['penuntut_umum'] = '; '.join(penuntut_list)
                    data['terdakwa'] = '; '.join(terdakwa_list)
            
            # --- BAGIAN B: Menangani Metadata Lainnya secara Dinamis ---
            elif len(tds) >= 2:
                key = tds[0].get_text(strip=True).lower().replace(' ', '_')
                val = tds[1].get_text(strip=True)
                if key in data:
                    data[key] = val
                    
    # 3. Mencari Link PDF (Revisi berdasarkan struktur HTML MA terbaru)
    pdf_found = False
    
    # Prioritaskan pencarian di sidebar bagian lampiran
    sidebar = soup.find('div', class_=lambda c: c and 'col-sm-4' in c)
    if sidebar:
        links = sidebar.find_all('a', href=True)
        for link in links:
            # Mencocokkan jika href mengandung '/pdf/' atau teks link diakhiri '.pdf'
            if '/pdf/' in link['href'] or link.get_text(strip=True).lower().endswith('.pdf'):
                data['url_pdf'] = link['href']
                pdf_found = True
                break
                
    # Jika tidak ketemu di sidebar, cari ke seluruh dokumen HTML
    if not pdf_found:
        all_links = soup.find_all('a', href=True)
        for link in all_links:
            if '/pdf/' in link['href'] or link.get_text(strip=True).lower().endswith('.pdf'):
                data['url_pdf'] = link['href']
                pdf_found = True
                break

    if not pdf_found:
        data['url_pdf'] = "Tidak ada PDF"
        
    return [data[f] for f in fields]

In [74]:
def main():
    warnings.filterwarnings('ignore')

    fileListURL = "listURLPNSurabayaPage41-60.txt"
    fileMetaCSV = "meta_putusan_pn_sby1.csv" 
    
    if not os.path.exists(fileListURL):
        print(f"[-] File {fileListURL} tidak ditemukan. Silakan jalankan skrip tahap 1 terlebih dahulu.")
        return

    with open(fileListURL, "r", encoding='utf-8') as f:
        bacaListURL = f.readlines()

    print("[+] Membuka Browser via DrissionPage...")
    page = init_drission_page()
    listHasil = []
    startTime = time.time()

    i = 1
    for barisURL in bacaListURL:
        url = barisURL.strip()
        if not url: 
            continue
            
        try:
            print(f"[+] Proses Row {i} : {url}")
            page.get(url)
            handle_cloudflare(page)
            time.sleep(2) 
            
            # --- PERUBAHAN PENTING: Passing 'url' sebagai current_url ke dalam fungsi ---
            hasil = generateMeta(page.html, current_url=url)
            listHasil.append(hasil)
            
        except Exception as e:
            print(f"   [-] Error Get Meta Info pada baris {i}: {e}")
        i += 1

    generateFileCSV(listHasil, fileMetaCSV)

    page.quit()
    endTime = time.time()
    print(f'Time Processing : {endTime-startTime:.2f} Second')

In [77]:
if __name__ == '__main__':
    main()

[+] Membuka Browser via DrissionPage...
[+] Proses Row 1 : https://putusan3.mahkamahagung.go.id/direktori/putusan/zaf1b334dfa995ec8ce9313434353039.html
[+] Proses Row 2 : https://putusan3.mahkamahagung.go.id/direktori/putusan/zaf1b332ef20583ca95f313433313136.html
[+] Proses Row 3 : https://putusan3.mahkamahagung.go.id/direktori/putusan/zaf1b334f8c56b508370313434353531.html
[+] Proses Row 4 : https://putusan3.mahkamahagung.go.id/direktori/putusan/zaf1b33503b6cb449e84313434363130.html
[+] Proses Row 5 : https://putusan3.mahkamahagung.go.id/direktori/putusan/zaf1b332c815d7f8c068313433303131.html
[+] Proses Row 6 : https://putusan3.mahkamahagung.go.id/direktori/putusan/zaf1b330fa17b64ca6e3313431373136.html
[+] Proses Row 7 : https://putusan3.mahkamahagung.go.id/direktori/putusan/zaf1b330f45f57329535313431373036.html
[+] Proses Row 8 : https://putusan3.mahkamahagung.go.id/direktori/putusan/zaf1b333010883e4b07f313433313436.html
[+] Proses Row 9 : https://putusan3.mahkamahagung.go.id/direktor